# **Data Cleaning**

## Objectives

* Correct all data errors via type correction, removal, or replacement as required

## Inputs

* outputs/datasets/collection/HotelBookings.csv

## Outputs

* Generate cleaned dataset saved as outputs/datasets/cleaned/HotelBookingsClean.csv

## Decisions from Revenue Manager

* Duplicate bookings should remain in place
* Bookings with no guests (adults, children or babies) should be removed
* Values > 4 for babies or children and > 2 for car park spaces should be replaced with their column's median value
* Bookings with >= 5 adults should be removed
* Isolated high and low outliers for adr should be removed
* Missing data in children should be replaced by 0
* Missing data in agent and company should be replaced by 0
* Missing data in country should be replaced by the mode value 

## Standard data cleaning

* Drop variables: `['reservation_status', 'reservation_status_date']`
* Perform dtype correction on children (float -> int)
* Ensure all numeric-as-categorical features are converted to 'category'


---

## Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [ ]:
import os
current_dir = os.getcwd()
current_dir

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chdir() defines the new current directory

In [ ]:
os.chdir(os.path.dirname(current_dir))

current_dir = os.getcwd()
current_dir


## Load Data

In [ ]:
import pandas as pd
df = pd.read_csv("outputs/datasets/collection/HotelBookings.csv")
df.head(3)

### Missingness

In [ ]:
# The following function is adapted from the 'Churnometer' walkthrough
def evaluate_missing_data(df):
    missing_data_absolute = df.isnull().sum()
    missing_data_percentage = round(missing_data_absolute/len(df)*100, 2)
    df_missing_data = (pd.DataFrame(
                            data={"RowsWithMissingData": missing_data_absolute,
                                   "PercentageOfDataset": missing_data_percentage,
                                   "DataType": df.dtypes}
                                    )
                          .sort_values(by=['PercentageOfDataset'], ascending=False)
                          .query("RowsWithMissingData > 0")
                          )

    return df_missing_data

In [ ]:
evaluate_missing_data(df)

* As per received guidance [Revenue Manager's Notebook](/jupyter_notebooks/03_rm_analysis.ipynb), replace missing values in `children`, `agent` and `company` with '0'

* Check that 0 is not in use for `agent` or `company`

In [ ]:
df[["company", "agent"]].describe()

* The minimum values confirm that 0 is not present in the existing values

In [ ]:
def replace_zeros(df):
    cols_to_replace_zero = ["children", "company", "agent"]
    for col in cols_to_replace_zero:
        df[col] = df[col].fillna(0)

replace_zeros(df)

In [ ]:
df.head(10)

* Re-evaluate missingness

In [ ]:
evaluate_missing_data(df)

* For `country` we should replace the missing values with the variable mode value

In [ ]:
df["country"].mode()

* Replace missing values with "PRT" and re-evaluate missingness

In [ ]:
df["country"] = df["country"].fillna("PRT")
evaluate_missing_data(df)

---

## Outliers

1. Bookings with no guests of any description should be removed

In [ ]:
df.shape

In [ ]:
no_guest_Df = df[((df["adults"] == 0) & (df["children"] == 0) & (df["babies"] == 0))]
no_guest_Df.shape

In [ ]:
df = df.drop(no_guest_Df.index)
df.shape

2. Bookings with 5 or more adults should be removed

In [ ]:
many_adults_df = df[df["adults"] >= 5]
many_adults_df.shape

In [ ]:
df = df.drop(many_adults_df.index)
df.shape

3. Bookings with more than 4 babies or children and bookings with more than 2 car park spaces should have the values replaced with the column's median value

In [ ]:
high_values_df = df[((df["children"] > 4) | (df["babies"] > 4) | (df["required_car_parking_spaces"] > 2))]
high_values_df               

* Replace all values above their threshold value with the column median value

* Ensure `children` and `babies` median values are whole numbers

In [ ]:
df[["children", "babies"]].median()

In [ ]:
def replace_median(df):
    column_thresholds = {"children": 4, "babies": 4, "required_car_parking_spaces":2}
    for col, threshold in column_thresholds.items():
        median_value = df[col].median()
        df.loc[df[col] > threshold, col] = median_value
    return df

df = replace_median(df)


In [ ]:
df.head()

* Re-run high values check to ensure changes have taken effect

In [ ]:
high_values_df = df[((df["children"] > 4) | (df["babies"] > 4) | (df["required_car_parking_spaces"] > 2))]
high_values_df

4. Remove high and low isolated extreme values for `adr`

In [ ]:
df["adr"].describe()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(18, 6))
sns.boxplot(data=df,
                x="reserved_room_type",
                y="adr",
                ax=ax)

plt.show()

* We can see from the chart that only one value exceeds 1000 for adr

In [ ]:
adr_extremes = df[((df["adr"] > 1000) | (df["adr"] < 0))]
adr_extremes

In [ ]:
df = df.drop(adr_extremes.index)
df["adr"].describe()

---

## Type Conversions

1. Children float64 should be int64 since partial children are not possible

In [ ]:
df["children"].dtypes

In [ ]:
df["children"] = df["children"].astype("int64")
df["children"].dtypes

2. Numeric-as-categoric conversion

In [ ]:
df.dtypes

* Convert `is_repeated_guest`, `agent` and `company` to categorical features 

In [ ]:
# First replace float64 dtypes with int64
float_list = ["agent", "company"]
for col in float_list:
    df[col] = df[col].astype("int64")

df.info()

In [ ]:
categorical_list = ["is_repeated_guest", "agent", "company"]
for col in categorical_list:
    df[col] = df[col].astype("category")

df.info()

* `is_canceled` is kept as numeric because the point-biserial correlation in the [correlation study](/jupyter_notebooks/05_correlation_study.ipynb) requires a numeric binary target
* `arrival_date_year` and `arrival_date_day_of_month` are kept as numeric for datetime amalgamation in [feature engineering](/jupyter_notebooks/06_feature_engineering.ipynb)
* `arrival_date_week_number` is kept as numeric due to its cyclical nature, a full categorisation can be performed in [feature engineering](/jupyter_notebooks/06_feature_engineering.ipynb)

---

## Drop data leakage columns

Since the model is intended to run on live hotel data, it should not be trained on any data only available *after* the booking's resolution. Two columns here provide data that is not available on live data: `reservation_status` and `reservation_status_date`.

* These 2 variables should be dropped

In [ ]:
df = df.drop(columns=["reservation_status", "reservation_status_date"])
df.head()

---

# Push files to Repo

* Save the cleaned dataframe for future use

In [ ]:
try:
  os.makedirs(name='outputs/datasets/cleaned')
except Exception as e:
  print(e)


In [ ]:
df.to_csv("outputs/datasets/cleaned/HotelBookingsClean.csv", index=False)

---

## Conclusions

* Compare cleaned dataset to original

In [ ]:
df_raw = pd.read_csv("outputs/datasets/collection/HotelBookings.csv")
df_raw.info()

In [ ]:
df.info()

In [ ]:
print(f"Original dataset had shape: {df_raw.shape}. The cleaned dataset has shape: {df.shape}")

* Missing data resolved: `children`, `agent`, `company` filled with 0; `country` filled with mode value ("PRT")
* Rows removed: no-guest bookings, bookings with 5+ adults, and isolated `adr` outliers totalling 198 rows removed (0.166% of original dataset) 
* Outlier values replaced with column median: `children`, `babies`, `required_car_parking_spaces`
* Dtype corrections applied: `children` to int64; `is_repeated_guest`, `agent`, `company` to category
* Leakage columns dropped: `reservation_status`, `reservation_status_date`
* Final dataset shape is: (119192, 30), saved to outputs/datasets/cleaned/HotelBookingsClean.csv

**Next Steps**
* The cleaned dataset is ready for the [Correlation Study](/jupyter_notebooks/05_correlation_study.ipynb), which will assess relationships between features and `is_canceled`